# AEGIS — Model A: YOLO11n Object Detector (fish / pipe / debris / structure)

Trains a 4-class underwater object detector per `AEGIS_Today_Scope.md`.

**Datasets (source-aware split, never random-frame split):**
- FathomNet API → `fish`, `structure`
- SeaClear (Kaggle mirror) → `debris`, `structure`
- SubPipe-Mini (Zenodo, direct download) → `pipe`

**Multi-session note:** Kaggle sessions cap at ~12h. This notebook does NOT use Ultralytics'
`resume=True` (that requires the same run folder to persist across sessions, which Kaggle does not
guarantee). Instead it uses a `CONTINUE_FROM_WEIGHTS` pattern: point it at a `last.pt` from a previous
session's output (re-attached as a Kaggle input dataset) and it starts a **new** run from those weights.

**If a dataset is not attached, cells below raise a clear `RuntimeError` telling you exactly what to
attach — they do not silently skip or fabricate data.**


In [ ]:
# --- Environment setup ---
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Kaggle's preinstalled PyTorch build is periodically upgraded to a newer CUDA toolchain that drops
# kernels for older GPU architectures (observed: Tesla P100 / sm_60 raising "no kernel image is
# available for execution on the device" on a stock Kaggle image -- PyTorch dropped Pascal/sm_60
# support starting at 2.8.0). Kaggle can assign a P100, T4 x2, or newer accelerator depending on
# availability, and we don't control which -- so reinstall the latest cu118 build that still ships
# sm_60 kernels (2.7.1, one release before the Pascal drop) BEFORE anything (including ultralytics,
# which pulls in torch) imports torch. An earlier attempt pinned torch==2.2.2, which also has sm_60 --
# but predates PyTorch's NumPy 2.0 support (added in 2.3), and forcibly downgrading numpy to match it
# broke a dozen other packages in Kaggle's base image compiled against NumPy 2.x's ABI (opencv-python,
# jax, cupy, etc. all crash). 2.7.1 supports both sm_60 and NumPy 2.x, so no numpy downgrade is needed.
gpu_present = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).returncode == 0
if gpu_present:
    print("GPU detected, installing a broadly architecture-compatible PyTorch build...")
    pip_install("torch==2.7.1", "torchvision==0.22.1", "--index-url", "https://download.pytorch.org/whl/cu118")
else:
    print("No GPU detected -- training will run on CPU and will be very slow.")

pip_install("ultralytics>=8.3.0", "fathomnet", "pycocotools", "opencv-python-headless", "onnx", "onnxslim", "remotezip")

import os, json, glob, shutil, random, hashlib, zipfile, urllib.request
from pathlib import Path
import numpy as np
import cv2
import torch

_du = shutil.disk_usage("/kaggle/working")
print(f"[resources] disk at startup: {_du.free/1e9:.1f} GB free / {_du.total/1e9:.1f} GB total")
print("Setup OK. CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (compute capability sm_{cap[0]}{cap[1]})")
    arch_str = f"sm_{cap[0]}{cap[1]}"
    if arch_str not in torch.cuda.get_arch_list():
        raise RuntimeError(
            f"Installed PyTorch does not include kernels for this GPU's architecture ({arch_str}). "
            f"Supported: {torch.cuda.get_arch_list()}. The pinned torch==2.7.1+cu118 install above should "
            "cover this -- if you still see this, Kaggle assigned a newer/older GPU than expected; "
            "check https://pytorch.org/get-started/locally/ for a wheel matching this GPU."
        )


In [ ]:
# --- Config ---
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ["fish", "pipe", "debris", "structure"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE = 384
EPOCHS_PER_SESSION = 130         # last run's 60 epochs finished in 0.87h -- plenty of session budget
                                  # left to reach the spec's 100-150 epoch target in one sitting
TOTAL_EPOCH_TARGET = 130         # spec calls for 100-150 total, across sessions
CHECKPOINT_PERIOD = 3            # save_period, per spec section 5

WORKING = Path("/kaggle/working")
DATA_ROOT = WORKING / "data"
YOLO_DATASET_DIR = WORKING / "yolo_dataset"
CHECKPOINT_OUT_DIR = WORKING / "checkpoints"
for p in (DATA_ROOT, YOLO_DATASET_DIR, CHECKPOINT_OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# --- CONTINUE_FROM_WEIGHTS pattern ---
# On session 1, this stays None and training starts from the pretrained yolo11n.pt.
# After a session ends, take /kaggle/working/checkpoints/last.pt from this notebook's Output,
# publish it as a new Kaggle Dataset version (e.g. "aegis-detector-ckpt"), attach that dataset
# as Input on the next run, and set CONTINUE_FROM_WEIGHTS to its path below.
CONTINUE_FROM_WEIGHTS = None
# Example: CONTINUE_FROM_WEIGHTS = "/kaggle/input/aegis-detector-ckpt/last.pt"

if CONTINUE_FROM_WEIGHTS is not None and not Path(CONTINUE_FROM_WEIGHTS).exists():
    raise RuntimeError(
        f"CONTINUE_FROM_WEIGHTS is set to {CONTINUE_FROM_WEIGHTS!r} but that file does not exist. "
        "Attach the checkpoint dataset as Input, or set CONTINUE_FROM_WEIGHTS = None to start fresh."
    )

print("Classes:", CLASS_TO_ID)
print("CONTINUE_FROM_WEIGHTS:", CONTINUE_FROM_WEIGHTS)


In [ ]:
# --- Helpers used by every dataset section ---

def require_path(path, instructions):
    """Raise a clear, actionable error if a required input path is missing."""
    p = Path(path)
    if not p.exists():
        raise RuntimeError(
            f"Required path not found: {path}\n\n{instructions}"
        )
    return p


def find_input_dir(name_fragments, base="/kaggle/input"):
    """Search /kaggle/input for a directory whose name contains all given fragments (case-insensitive)."""
    base = Path(base)
    if not base.exists():
        return None
    for child in base.iterdir():
        low = child.name.lower()
        if all(f.lower() in low for f in name_fragments):
            return child
    return None


def log_resources(tag=""):
    """Print disk and memory headroom so a failure's true cause (e.g. disk exhaustion) is visible in
    the log well before the crash, not just as an opaque OSError at the point of failure."""
    du = shutil.disk_usage("/kaggle/working")
    free_gb, used_gb, total_gb = du.free / 1e9, du.used / 1e9, du.total / 1e9
    mem_note = ""
    try:
        meminfo = {}
        with open("/proc/meminfo") as f:
            for line in f:
                k, v = line.split(":", 1)
                meminfo[k.strip()] = v.strip()
        mem_total_gb = int(meminfo["MemTotal"].split()[0]) / 1e6
        mem_avail_gb = int(meminfo["MemAvailable"].split()[0]) / 1e6
        mem_note = f" | mem: {mem_avail_gb:.1f}/{mem_total_gb:.1f} GB available"
    except Exception:
        pass
    label = f" [{tag}]" if tag else ""
    print(f"[resources{label}] disk: {free_gb:.1f} GB free / {total_gb:.1f} GB total (used {used_gb:.1f} GB){mem_note}")


def require_free_disk_gb(min_gb, tag=""):
    """Fail loudly and early if there isn't enough headroom for a known-large step, instead of
    crashing deep inside an extraction/download with an opaque OSError."""
    free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
    if free_gb < min_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB free on /kaggle/working, need at least {min_gb} GB for {tag or 'the next step'}. "
            "Free up space (delete unneeded files under /kaggle/working) or reduce this step's footprint "
            "before retrying -- proceeding would very likely crash with 'No space left on device'."
        )


MAX_IMAGE_DIM = 640  # training runs at IMG_SIZE=384; storing source images at native resolution
                      # (multi-megapixel deep-sea/GoPro photos) wastes disk for no training benefit --
                      # this cap sits comfortably above IMG_SIZE for augmentation/letterbox headroom.


def save_resized_image(image_bytes, dst_path, max_dim=MAX_IMAGE_DIM, quality=90):
    """Decode, downscale (if needed) to max_dim on the long side, and write as JPEG. Returns True on
    success. Every image source in this notebook funnels through this instead of writing raw
    downloaded/extracted bytes straight to disk, since that is what exhausted /kaggle/working's ~20GB
    quota on earlier runs (FathomNet alone used 11.1GB for ~2,750 native-resolution images)."""
    arr = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)
    if arr is None:
        return False
    h, w = arr.shape[:2]
    long_side = max(h, w)
    if long_side > max_dim:
        scale = max_dim / long_side
        arr = cv2.resize(arr, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", arr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        return False
    dst_path.write_bytes(buf.tobytes())
    return True


# samples accumulate here: list of dicts {image: Path, label_lines: [str,...], source_group: str}
ALL_SAMPLES = []
MANIFEST = []  # dataset manifest entries, per scope section 4


def yolo_line(class_id, x, y, w, h, img_w, img_h):
    """Convert an absolute pixel box (x,y,w,h from top-left) to a normalized YOLO label line."""
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    nw = w / img_w
    nh = h / img_h
    cx, cy, nw, nh = (min(max(v, 0.0), 1.0) for v in (cx, cy, nw, nh))
    return f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"

print("Helpers ready")
log_resources("after setup")


## Section 1 — FathomNet (`fish`, `structure`)

Public API, no manual attach needed. Uses concept-level queries verified live against the FathomNet database (counts as of this build: `bony fish` ~32k boxes, structure-ish concepts ~1.7k boxes combined — FathomNet is fish-heavy and structure-sparse, which is expected and reflected in the manifest, not padded).

In [ ]:
# --- FathomNet download ---
from fathomnet.api import images as fn_images, boundingboxes as fn_boxes

FATHOMNET_CONCEPTS = {
    # concept name (exact FathomNet taxonomy/darwin-core string) -> our class
    "bony fish": "fish",
    "ratfish": "fish",
    "jawless fish": "fish",
    "ship wreckage": "structure",
    "anchor": "structure",
    "mooring": "structure",
    "backscatter mooring": "structure",
    "equipment": "structure",
    "Benthic Instrument Node": "structure",
    "mini container array": "structure",
    "ship container": "structure",
    "site marker": "structure",
    "cable": "structure",
    "cable spool": "structure",
}

# Cap per-concept downloads to keep the one-day compute/bandwidth budget sane. bony fish alone has
# ~32k boxes; we do not need all of them for a 4-class MVP detector.
MAX_IMAGES_PER_CONCEPT = 1200

FATHOMNET_DIR = DATA_ROOT / "fathomnet"
(FATHOMNET_DIR / "images").mkdir(parents=True, exist_ok=True)
(FATHOMNET_DIR / "labels").mkdir(parents=True, exist_ok=True)

fathomnet_used = {}
fathomnet_failed = 0

for concept, cls in FATHOMNET_CONCEPTS.items():
    try:
        recs = fn_images.find_by_concept(concept)
    except Exception as e:
        print(f"  [WARN] FathomNet query failed for concept={concept!r}: {e}")
        continue
    if not recs:
        print(f"  [WARN] No FathomNet images returned for concept={concept!r}")
        continue
    random.Random(SEED).shuffle(recs)
    recs = recs[:MAX_IMAGES_PER_CONCEPT]
    fathomnet_used[concept] = 0

    for rec in recs:
        if not rec.url or not rec.width or not rec.height:
            continue
        uid = rec.uuid or hashlib.sha1(rec.url.encode()).hexdigest()[:16]
        img_path = FATHOMNET_DIR / "images" / f"fn_{uid}.jpg"
        lbl_path = FATHOMNET_DIR / "labels" / f"fn_{uid}.txt"
        if not img_path.exists():
            try:
                with urllib.request.urlopen(rec.url, timeout=30) as resp:
                    image_bytes = resp.read()
                # bbox coords below stay normalized against the ORIGINAL rec.width/rec.height, so
                # downscaling the stored copy doesn't affect label correctness -- only disk footprint.
                if not save_resized_image(image_bytes, img_path):
                    fathomnet_failed += 1
                    continue
            except Exception:
                fathomnet_failed += 1
                continue

        lines = []
        for box in (rec.boundingBoxes or []):
            box_cls = FATHOMNET_CONCEPTS.get(box.concept)
            if box_cls is None:
                continue  # a box for a concept we didn't request/map -> not one of our 4 classes
            if box.x is None or box.width is None:
                continue
            lines.append(yolo_line(CLASS_TO_ID[box_cls], box.x, box.y, box.width, box.height, rec.width, rec.height))
        if not lines:
            # Image returned for this concept query but had no mappable boxes on it -> skip, don't fabricate.
            img_path.unlink(missing_ok=True)
            continue
        lbl_path.write_text("\n".join(lines))
        ALL_SAMPLES.append({"image": img_path, "label_lines": lines, "source_group": f"fathomnet_{concept}"})
        fathomnet_used[concept] += 1

print("FathomNet images kept per concept:", fathomnet_used)
print("FathomNet download failures:", fathomnet_failed)
log_resources("after FathomNet")

if sum(fathomnet_used.values()) == 0:
    raise RuntimeError(
        "FathomNet download returned zero usable images. Check network/internet access is enabled "
        "for this kernel (Settings -> Internet -> On) and that the fathomnet package installed correctly."
    )

MANIFEST.append({
    "dataset": "FathomNet",
    "source": "https://fathomnet.org (fathomnet-py API)",
    "license": "Item-level licensing; verify per downloaded subset before any redistribution",
    "class_mapping": FATHOMNET_CONCEPTS,
    "images_used": sum(fathomnet_used.values()),
})


## Section 2 — SeaClear (`debris`, `structure`)

**Manual step required:** attach the Kaggle dataset `jocelyndumlao/seaclear-marine-debris-detection-and-segmentation` as an Input to this kernel (Add Input -> search "Seaclear Marine Debris"). This is a Kaggle-hosted mirror of the TU Delft SeaClear dataset; annotations are a single COCO-format JSON file per the dataset's own description.

In [ ]:
# --- SeaClear ---
SEACLEAR_DIR = find_input_dir(["seaclear"])
if SEACLEAR_DIR is None:
    raise RuntimeError(
        "SeaClear dataset not found under /kaggle/input. Attach it: on Kaggle, click 'Add Input', "
        "search for 'Seaclear Marine Debris Detection and Segmentation' "
        "(dataset slug: jocelyndumlao/seaclear-marine-debris-detection-and-segmentation), and attach it "
        "to this kernel, then re-run this cell."
    )

coco_json_candidates = list(SEACLEAR_DIR.rglob("*.json"))
if not coco_json_candidates:
    raise RuntimeError(
        f"SeaClear dataset is attached at {SEACLEAR_DIR} but no .json (COCO annotation) file was found "
        "inside it. The dataset's own description states annotations ship as a COCO-format JSON file; "
        "if the mirror changed, download the original from "
        "https://data.4tu.nl/datasets/4f1dff25-e157-4399-a5d4-478055461689 instead and attach that."
    )
coco_path = max(coco_json_candidates, key=lambda p: p.stat().st_size)  # biggest json = the annotation file
print("Using SeaClear COCO annotations:", coco_path)

with open(coco_path) as f:
    coco = json.load(f)

# 40 raw SeaClear categories are litter/animal/plant/robot-part mixes. Map by keyword; anything
# unmapped is logged and dropped (background), never guessed into a class.
DEBRIS_KEYWORDS = ["plastic", "bottle", "can", "bag", "metal", "glass", "rubber", "cloth", "fabric",
                   "fishing", "net", "rope", "trash", "litter", "tire", "tyre", "cable", "wire",
                   "paper", "wood", "styrofoam", "foam", "clothing", "shoe", "container"]
STRUCTURE_KEYWORDS = ["rov", "robot", "auv", "frame", "structure", "part", "propeller", "thruster"]

def map_seaclear_category(name):
    low = name.lower()
    if any(k in low for k in STRUCTURE_KEYWORDS):
        return "structure"
    if any(k in low for k in DEBRIS_KEYWORDS):
        return "debris"
    return None

cat_id_to_class = {}
unmapped_cats = []
for cat in coco["categories"]:
    mapped = map_seaclear_category(cat["name"])
    cat_id_to_class[cat["id"]] = mapped
    if mapped is None:
        unmapped_cats.append(cat["name"])

print(f"SeaClear categories mapped: {len(cat_id_to_class) - len(unmapped_cats)} / {len(cat_id_to_class)}")
if unmapped_cats:
    print("SeaClear categories dropped as background (not fish/pipe/debris/structure):", unmapped_cats)

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = {}
for ann in coco["annotations"]:
    anns_by_image.setdefault(ann["image_id"], []).append(ann)

SEACLEAR_LABELS_DIR = DATA_ROOT / "seaclear" / "labels"
SEACLEAR_LABELS_DIR.mkdir(parents=True, exist_ok=True)

seaclear_used = 0
seaclear_by_class = {"debris": 0, "structure": 0}
for img_id, img_rec in images_by_id.items():
    anns = anns_by_image.get(img_id, [])
    lines = []
    for ann in anns:
        cls = cat_id_to_class.get(ann["category_id"])
        if cls is None:
            continue
        x, y, w, h = ann["bbox"]
        lines.append(yolo_line(CLASS_TO_ID[cls], x, y, w, h, img_rec["width"], img_rec["height"]))
        seaclear_by_class[cls] += 1
    if not lines:
        continue
    # image path: file_name is relative to the "Seaclear Marine Debris Dataset" folder in this mirror
    candidates = list(SEACLEAR_DIR.rglob(Path(img_rec["file_name"]).name))
    if not candidates:
        continue
    img_path = candidates[0]
    # source_group = site/camera folder (parent dir of the image) -> source-aware split unit
    source_group = f"seaclear_{img_path.parent.parent.name}_{img_path.parent.name}"
    lbl_path = SEACLEAR_LABELS_DIR / (img_path.stem + ".txt")
    lbl_path.write_text("\n".join(lines))
    ALL_SAMPLES.append({"image": img_path, "label_lines": lines, "source_group": source_group})
    seaclear_used += 1

print("SeaClear images used:", seaclear_used, "| boxes by class:", seaclear_by_class)
log_resources("after SeaClear")
if seaclear_used == 0:
    raise RuntimeError(
        "SeaClear COCO annotations parsed but produced zero usable (image, label) pairs. Check that "
        "image file_name values in the JSON actually match filenames under the attached input directory."
    )

MANIFEST.append({
    "dataset": "SeaClear Marine Debris Detection and Segmentation (Kaggle mirror)",
    "source": "kaggle.com/datasets/jocelyndumlao/seaclear-marine-debris-detection-and-segmentation "
              "(mirror of TU Delft / 4TU research data)",
    "license": "CC0-1.0 as listed on the Kaggle mirror (original TU Delft release is CC BY 4.0 — "
               "attribute the original SeaClear project regardless of which license governs the mirror)",
    "class_mapping": {"categories matching debris keywords": "debris",
                       "categories matching structure keywords": "structure",
                       "everything else (fish/plant/animal/other)": "background (dropped)"},
    "images_used": seaclear_used,
})


## Section 3 — SubPipe-Mini (`pipe`)

Direct download from Zenodo (no login required, internet access must be enabled on this kernel).

**Important deviation from the scope doc, verified during implementation — see the chat message
for full detail:** SubPipe's YOLO-format detection annotations exist only for its **side-scan sonar**
images, which are out of scope today. The RGB (GoPro) camera images only ship **segmentation masks**
(pipe-pixel vs background), not bounding boxes. To get a `pipe` class for the RGB detector without
touching sonar, this notebook derives bounding boxes from the segmentation masks via connected-component
analysis. This is real pixel data, not fabricated — but it's a different pipeline than "download
pre-made YOLO labels," so flagging it explicitly rather than doing it silently.

In [ ]:
# --- SubPipe-Mini: range-request extraction, no full zip download ---
# Rewritten twice: first a bulk extractall() blew the disk (Cam0/Cam1 hold 18k+ unlabeled raw frames,
# several GB). Filtering to just those two folders still meant downloading the full 6.1GB zip. Then
# live inspection of the actual archive (via remotezip, reading only the central directory over HTTP
# range requests -- no download needed to just list contents) revealed the real layout doesn't match
# what the SubPipe GitHub README describes for the full dataset: labeled pairs live in a single flat
# "Segmentation/" folder (not nested under Cam0_images/Cam1_images), pairing "<ts>.jpg" with
# "<ts>_label.png" -- 1,295 members / ~343MB total, vs 6.1GB for the whole archive. Cam0_images/
# Cam1_images turned out to be 18,332 completely unlabeled frames we never needed at all.
# remotezip lets us fetch only the Segmentation/ member bytes we actually need over HTTP range
# requests, so this never downloads the other 5.8GB of the zip in the first place.
from remotezip import RemoteZip

SUBPIPE_URL = "https://zenodo.org/records/12666132/files/SubPipeMini.zip?download=1"
SUBPIPE_IMAGES_DIR = DATA_ROOT / "subpipe" / "images"
SUBPIPE_LABELS_DIR = DATA_ROOT / "subpipe" / "labels"
SUBPIPE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
SUBPIPE_LABELS_DIR.mkdir(parents=True, exist_ok=True)

MIN_PIPE_PIXELS = 50  # ignore microscopic noise blobs in the mask

subpipe_used = 0
subpipe_boxes = 0
subpipe_skipped_no_pair = 0

try:
    zf = RemoteZip(SUBPIPE_URL)
except Exception as e:
    raise RuntimeError(
        f"Failed to open SubPipeMini.zip for remote reading from {SUBPIPE_URL}: {e}\n"
        "Check that Internet access is enabled for this kernel (Settings -> Internet -> On). "
        "If Zenodo has moved this record, check https://zenodo.org/records/12666132 manually."
    )

with zf:
    seg_members = [m for m in zf.namelist() if "/segmentation/" in m.lower()]
    if not seg_members:
        raise RuntimeError(
            "SubPipeMini.zip has no member under a 'Segmentation/' folder. The archive layout has "
            "changed from what this notebook assumes -- list zf.namelist() manually and update the "
            "matching logic below."
        )
    label_members = [m for m in seg_members if m.lower().endswith("_label.png")]
    if not label_members:
        raise RuntimeError(
            f"SubPipeMini.zip's Segmentation/ folder has {len(seg_members)} members but none end in "
            "'_label.png'. The archive layout has changed -- inspect zf.namelist() manually."
        )
    print(f"Found {len(label_members)} SubPipe segmentation masks in the archive's Segmentation/ "
          f"folder (fetching only these ~343MB over HTTP range requests, not the full 6.1GB zip)")

    # basename -> full member path, for O(1) original-image lookup per mask
    image_members_by_basename = {
        Path(m).name: m for m in seg_members if not m.lower().endswith("_label.png")
    }

    for i, label_member in enumerate(label_members):
        orig_name = Path(label_member).name.replace("_label.png", ".jpg")  # masks pair with .jpg, not .png
        orig_member = image_members_by_basename.get(orig_name)
        if orig_member is None:
            subpipe_skipped_no_pair += 1
            continue

        mask_bytes = zf.read(label_member)  # decoded in memory, never written to disk
        mask = cv2.imdecode(np.frombuffer(mask_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        img_h, img_w = mask.shape[:2]
        binary = (mask > 0).astype(np.uint8)

        lines = []
        if binary.sum() >= MIN_PIPE_PIXELS:
            num_labels, comp = cv2.connectedComponents(binary)
            for comp_id in range(1, num_labels):
                ys, xs = np.where(comp == comp_id)
                if len(xs) < MIN_PIPE_PIXELS:
                    continue
                x0, x1 = xs.min(), xs.max()
                y0, y1 = ys.min(), ys.max()
                lines.append(yolo_line(CLASS_TO_ID["pipe"], x0, y0, x1 - x0, y1 - y0, img_w, img_h))
                subpipe_boxes += 1

        # Boxes above are normalized against the mask's own img_w/img_h, so downscaling the stored
        # copy doesn't affect label correctness, only disk footprint.
        uid = hashlib.sha1(orig_member.encode()).hexdigest()[:16]
        dst_img = SUBPIPE_IMAGES_DIR / f"subpipe_{uid}.jpg"
        if not dst_img.exists():
            try:
                orig_bytes = zf.read(orig_member)
                if not save_resized_image(orig_bytes, dst_img):
                    continue
            except OSError as e:
                log_resources("SubPipe extraction failure")
                raise RuntimeError(f"Ran out of disk space extracting {orig_member} from SubPipeMini.zip: {e}")

        # images with zero pipe pixels are kept as valid negatives (empty label file) -> helps precision
        lbl_path = SUBPIPE_LABELS_DIR / (dst_img.stem + ".txt")
        lbl_path.write_text("\n".join(lines))
        # Segmentation/ is one continuous recording (filenames are unix timestamps), not multiple
        # sites/videos -- treating it as a single source-aware group would dump 100% of SubPipe into
        # one split. Bucket by a 30s time window instead: adjacent (near-duplicate) frames stay
        # together, avoiding the leakage the source-aware rule exists to prevent, while still giving
        # the split multiple independent chunks to draw from.
        try:
            ts = float(Path(orig_member).stem)
            source_group = f"subpipe_seg_t{int(ts // 30)}"
        except ValueError:
            source_group = "subpipe_segmentation"
        ALL_SAMPLES.append({"image": dst_img, "label_lines": lines, "source_group": source_group})
        subpipe_used += 1

        if (i + 1) % 200 == 0 or (i + 1) == len(label_members):
            log_resources(f"SubPipe {i + 1}/{len(label_members)} masks processed")

log_resources("after SubPipe extraction")

print(f"SubPipe images used: {subpipe_used} | pipe boxes derived from masks: {subpipe_boxes} | "
      f"masks with no matching original image (skipped): {subpipe_skipped_no_pair}")
if subpipe_used == 0:
    raise RuntimeError(
        "SubPipe-Mini archive opened but zero (image, mask) pairs were matched. The naming "
        "convention has likely diverged from '<ts>.jpg' / '<ts>_label.png' -- inspect "
        "zf.namelist() manually and update the matching logic above."
    )

MANIFEST.append({
    "dataset": "SubPipe-Mini",
    "source": "https://zenodo.org/records/12666132 (direct download, OceanScan-MST)",
    "license": "CC-BY-4.0 per Zenodo record; explicit reuse text tied to OceanScan-MST -- flag for "
               "legal review before any commercial use",
    "class_mapping": {"segmentation mask pipe-pixels (connected components)": "pipe"},
    "images_used": subpipe_used,
    "note": "Bounding boxes derived from RGB segmentation masks, not native YOLO labels -- see markdown above.",
})


## Section 4 — Source-aware train/val/test split

Splits by `source_group` (dataset+site/camera/concept), never by random frame shuffle, per scope section 4 non-negotiable #1.

In [ ]:
# --- Source-aware split ---
if not ALL_SAMPLES:
    raise RuntimeError("No samples collected from any dataset -- cannot build a split. See errors above.")

groups = sorted(set(s["source_group"] for s in ALL_SAMPLES))
rng = random.Random(SEED)
rng.shuffle(groups)

# Class-stratified on top of source-aware: a pure random group shuffle can (and did, in an earlier
# run) leave a whole class with zero instances in test/val purely by luck of which groups landed
# where. Reserve one group containing each class for test and val first (when at least two distinct
# groups carry that class -- never split a single group's images across splits to do this), then fill
# the remaining ~15%/15%/70% target sizes from what's left over.
group_classes = {}
for s in ALL_SAMPLES:
    classes_in_sample = {CLASSES[int(line.split()[0])] for line in s["label_lines"]}
    group_classes.setdefault(s["source_group"], set()).update(classes_in_sample)

class_to_groups = {c: [g for g in groups if c in group_classes.get(g, set())] for c in CLASSES}

test_groups, val_groups = set(), set()
for c in CLASSES:
    candidates = [g for g in class_to_groups[c] if g not in test_groups and g not in val_groups]
    if len(candidates) >= 2:
        test_groups.add(candidates[0])
        val_groups.add(candidates[1])
    elif len(candidates) == 1:
        # Only one group anywhere has this class -- put it in test (held-out eval matters most);
        # val will miss this class, same limitation as before but now at least test doesn't.
        test_groups.add(candidates[0])

n = len(groups)
n_test_target = max(1, round(n * 0.15))
n_val_target = max(1, round(n * 0.15))
remaining = [g for g in groups if g not in test_groups and g not in val_groups]
test_groups.update(remaining[:max(0, n_test_target - len(test_groups))])
remaining = [g for g in remaining if g not in test_groups]
val_groups.update(remaining[:max(0, n_val_target - len(val_groups))])

def split_of(group):
    if group in test_groups:
        return "test"
    if group in val_groups:
        return "val"
    return "train"

for c in CLASSES:
    covered = {split_of(g) for g in class_to_groups[c]} if class_to_groups[c] else set()
    missing = {"train", "val", "test"} - covered
    if missing:
        print(f"[WARN] class {c!r} has no source group in: {sorted(missing)} "
              f"-- only {len(class_to_groups[c])} distinct group(s) contain it in the whole dataset.")

for split in ("train", "val", "test"):
    (YOLO_DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

split_class_counts = {s: {c: 0 for c in CLASSES} for s in ("train", "val", "test")}
split_image_counts = {"train": 0, "val": 0, "test": 0}

for sample in ALL_SAMPLES:
    split = split_of(sample["source_group"])
    img_src = sample["image"]
    dst_img = YOLO_DATASET_DIR / split / "images" / f"{hashlib.sha1(str(img_src).encode()).hexdigest()[:12]}{img_src.suffix}"
    dst_lbl = dst_img.with_suffix(".txt").parent.parent / "labels" / dst_img.with_suffix(".txt").name
    if not dst_img.exists():
        try:
            os.link(img_src, dst_img)  # hardlink: free, works for FathomNet/SubPipe (same filesystem,
                                        # already downscaled at save time -- see save_resized_image)
        except OSError:
            # Cross-filesystem source (e.g. SeaClear, mounted read-only under /kaggle/input) --
            # os.link can't span filesystems. Fall back to a resized copy, not a raw shutil.copy2:
            # SeaClear ships at 1920x1080, and a full-res copy of ~7,000 images would repeat the same
            # disk blowout FathomNet caused before it was fixed to downscale at download time.
            if not save_resized_image(img_src.read_bytes(), dst_img):
                shutil.copy2(img_src, dst_img)  # last-resort: keep the sample rather than drop it
    dst_lbl.write_text("\n".join(sample["label_lines"]))
    split_image_counts[split] += 1
    for line in sample["label_lines"]:
        cls_id = int(line.split()[0])
        split_class_counts[split][CLASSES[cls_id]] += 1

print("Images per split:", split_image_counts)
print("Box counts per split/class:")
for split in ("train", "val", "test"):
    print(" ", split, split_class_counts[split])

for split in ("train", "val", "test"):
    if split_image_counts[split] == 0:
        raise RuntimeError(f"Split {split!r} ended up with zero images -- widen the dataset sources or adjust split ratios.")

DATA_YAML = YOLO_DATASET_DIR / "data.yaml"
DATA_YAML.write_text(
    f"path: {YOLO_DATASET_DIR}\n"
    f"train: train/images\n"
    f"val: val/images\n"
    f"test: test/images\n"
    f"names:\n" + "\n".join(f"  {i}: {c}" for i, c in enumerate(CLASSES)) + "\n"
)
print("Wrote", DATA_YAML)
log_resources("after split")


## Section 5 — Dataset manifest (scope section 4, non-negotiable #3)

In [ ]:
manifest_path = WORKING / "dataset_manifest.json"
manifest_path.write_text(json.dumps(MANIFEST, indent=2, default=str))
for entry in MANIFEST:
    print(f"- {entry['dataset']}: {entry['images_used']} images | license: {entry['license']}")
print("\nFull manifest written to", manifest_path)


## Section 6 — Train YOLO11n

In [ ]:
from ultralytics import YOLO

start_weights = CONTINUE_FROM_WEIGHTS if CONTINUE_FROM_WEIGHTS else "yolo11n.pt"
print("Starting weights:", start_weights)
log_resources("before training")
model = YOLO(start_weights)

RUN_NAME = "aegis_detector"
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_PER_SESSION,
    imgsz=IMG_SIZE,
    seed=SEED,
    save_period=CHECKPOINT_PERIOD,
    project=str(WORKING / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    resume=False,   # deliberate: see CONTINUE_FROM_WEIGHTS note at top, not Ultralytics' resume=True
    patience=0,     # multi-session training: let it use the full epoch budget every session
)

run_dir = WORKING / "runs" / RUN_NAME
last_pt = run_dir / "weights" / "last.pt"
best_pt = run_dir / "weights" / "best.pt"
for src in (last_pt, best_pt):
    if src.exists():
        shutil.copy2(src, CHECKPOINT_OUT_DIR / src.name)
print("Checkpoints copied to", CHECKPOINT_OUT_DIR)
print(
    "\nTo continue training in a NEW session: publish", CHECKPOINT_OUT_DIR,
    "as a Kaggle Dataset (e.g. aegis-detector-ckpt), attach it as Input next run, "
    "and set CONTINUE_FROM_WEIGHTS to its last.pt path in the Config cell above."
)


## Section 7 — Evaluate on held-out test split (never used during training)

In [ ]:
best_weights = CHECKPOINT_OUT_DIR / "best.pt"
eval_model = YOLO(str(best_weights if best_weights.exists() else CHECKPOINT_OUT_DIR / "last.pt"))

metrics = eval_model.val(data=str(DATA_YAML), split="test", imgsz=IMG_SIZE)

# metrics.box.{p,r,ap50,ap} are indexed by whichever classes actually had >=1 ground-truth instance
# in the test split -- NOT by CLASSES position. A class absent from test (as happened to 'fish' in an
# earlier run) shrinks these arrays, silently misaligning a naive zip(CLASSES, box.r). Use the
# officially-provided ap_class_index to map each entry back to its real class, and report absent
# classes explicitly as null rather than mislabeling a different class's numbers onto them.
present_class_ids = [int(i) for i in metrics.box.ap_class_index]

# support (ground-truth box count per class in the test split) -- pulled from split_class_counts
# computed in Section 4, NOT from Ultralytics internals: metrics.box/validator do not reliably expose
# a stable per-class instance-count attribute across versions (verified: eval_model.validator.nt_per_class
# came back unset in practice), so trust the counts this notebook already derived from its own labels.
per_class = {c: None for c in CLASSES}
for pos, cls_id in enumerate(present_class_ids):
    per_class[CLASSES[cls_id]] = {
        "precision": float(metrics.box.p[pos]),
        "recall": float(metrics.box.r[pos]),
        "mAP50": float(metrics.box.ap50[pos]),
        "mAP50-95": float(metrics.box.ap[pos]),
        "support": int(split_class_counts["test"][CLASSES[cls_id]]),
    }
missing_classes = [c for c, v in per_class.items() if v is None]
if missing_classes:
    print(f"[NOTE] classes with zero test-split instances (not evaluated): {missing_classes}")

# metrics.box.map50/map/mp/mr are Ultralytics' own means over present_class_ids only -- i.e. already
# excluding zero-support classes, consistent with how the classifier report's macro_f1 is now
# computed. Recorded explicitly here (rather than just trusting the attribute name) so this report is
# self-documenting about what the headline numbers do and do not include.
eval_report = {
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
    "precision_mean": float(metrics.box.mp),
    "recall_mean": float(metrics.box.mr),
    "per_class": per_class,
    "note": "Headline metrics above are averaged only over classes with test-split support > 0 "
            "(see per-class 'support'); classes with zero instances are reported as null, not averaged in.",
}
print(json.dumps(eval_report, indent=2))
(WORKING / "detector_eval_report.json").write_text(json.dumps(eval_report, indent=2))


## Section 8 — Export ONNX

In [ ]:
onnx_path = eval_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
print("Exported ONNX model to:", onnx_path)
